# 00. Diagnóstico para Reestruturação a Nível de Paciente

## Objetivo
Este notebook investiga a estrutura do dataset COVIDx CT-3A para responder às perguntas críticas antes de reestruturar o projeto para tratamento **a nível de paciente** (em vez de a nível de slice).

### O que vamos verificar:
1. **Existência e conteúdo do `metadata.csv`** — ele contém o `patient id`?
2. **Formato dos arquivos de anotação** (`.txt`) — quais colunas existem?
3. **Padrão dos nomes dos arquivos** — é possível extrair o patient_id do filename?
4. **Mapeamento filename → patient_id** — qual a melhor estratégia?
5. **Overlap de pacientes entre train/test** — o split original já é patient-disjoint?
6. **Distribuição de slices por paciente** — quantos slices cada paciente tem?
7. **Impacto no split atual** — os arquivos `_filtered.txt` têm data leakage?

In [ ]:
import sys
import os

# Adiciona a raiz do projeto ao path
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
import re

from src.config import Config

BASE = Path(Config.BASE_PATH)
print(f"📂 DATASET_PATH: {BASE}")
print(f"📂 Existe: {BASE.exists()}")

---
## 1. Inventário de Arquivos no Dataset
Listamos TODOS os arquivos `.txt`, `.csv` e pastas de imagens disponíveis.

In [ ]:
print("=" * 60)
print("📄 ARQUIVOS DE ANOTAÇÃO E METADADOS")
print("=" * 60)

# Listar todos os .txt e .csv na raiz do dataset
for ext in ['*.txt', '*.csv']:
    for f in sorted(BASE.glob(ext)):
        size_kb = f.stat().st_size / 1024
        lines = sum(1 for _ in open(f, encoding='utf-8', errors='ignore'))
        print(f"  [{ext[1:].upper():>4}] {f.name:<40} ({lines:>6} linhas, {size_kb:.1f} KB)")

print()
print("=" * 60)
print("📁 PASTAS DE IMAGENS")
print("=" * 60)

for d in sorted(BASE.iterdir()):
    if d.is_dir():
        # Conta arquivos e subpastas
        n_files = sum(1 for _ in d.rglob('*') if _.is_file())
        print(f"  {d.name:<45} ({n_files:>6} arquivos)")

---
## 2. Verificação do `metadata.csv`
O arquivo `metadata.csv` do COVIDx CT-3A contém a coluna `patient id` que mapeia cada filename para o paciente. Vamos verificar se ele existe e inspecionar seu conteúdo.

In [ ]:
metadata_path = BASE / 'metadata.csv'
has_metadata = metadata_path.exists()

print(f"🔍 metadata.csv existe? {'✅ SIM' if has_metadata else '❌ NÃO'}")

if has_metadata:
    metadata = pd.read_csv(metadata_path)
    print(f"\n📊 Shape: {metadata.shape}")
    print(f"\n📋 Colunas: {list(metadata.columns)}")
    print(f"\n🔎 Primeiras 10 linhas:")
    display(metadata.head(10))
    
    # Verifica se tem a coluna patient id
    patient_col = None
    for col in metadata.columns:
        if 'patient' in col.lower() and 'id' in col.lower():
            patient_col = col
            break
    
    if patient_col:
        n_patients = metadata[patient_col].nunique()
        print(f"\n✅ Coluna de paciente encontrada: '{patient_col}'")
        print(f"   Total de pacientes únicos: {n_patients}")
        print(f"   Total de registros: {len(metadata)}")
        print(f"   Média de slices por paciente: {len(metadata) / n_patients:.1f}")
    else:
        print("\n⚠️ Coluna 'patient id' NÃO encontrada nas colunas!")
        print("   Colunas disponíveis:", list(metadata.columns))
else:
    print("\n⚠️ O arquivo metadata.csv não foi encontrado no diretório do dataset.")
    print("   Vamos tentar extrair o patient_id do nome dos arquivos na próxima seção.")

---
## 3. Formato dos Arquivos de Anotação Originais
Vamos inspecionar os arquivos `.txt` originais do COVIDx CT-3A para entender o formato.

In [ ]:
# Busca os arquivos de anotação originais
original_txts = {
    'train_original': BASE / 'train_COVIDx_CT-3A.txt',
    'val_original':   BASE / 'val_COVIDx_CT-3A.txt',
    'test_original':  BASE / 'test_COVIDx_CT-3A.txt',
}

# Busca os arquivos filtrados (gerados pelo projeto atual)
filtered_txts = {
    'train_filtered': BASE / 'train_filtered.txt',
    'val_filtered':   BASE / 'val_filtered.txt',
    'test_filtered':  BASE / 'test_filtered.txt',
}

print("=" * 60)
print("ARQUIVOS DE ANOTAÇÃO ORIGINAIS")
print("=" * 60)

for name, path in original_txts.items():
    print(f"\n{'─'*40}")
    print(f"📄 {name}: {'✅ Existe' if path.exists() else '❌ Não encontrado'}")
    if path.exists():
        with open(path) as f:
            lines = f.readlines()
        print(f"   Total de linhas: {len(lines)}")
        print(f"   Formato (primeiras 5 linhas):")
        for line in lines[:5]:
            parts = line.strip().split()
            print(f"     {parts}  ({len(parts)} campos)")
        
        # Analisa o número de campos
        n_fields = [len(line.strip().split()) for line in lines[:100]]
        print(f"   Campos por linha: {Counter(n_fields)}")

print(f"\n{'='*60}")
print("ARQUIVOS FILTRADOS (gerados pelo projeto)")
print("=" * 60)

for name, path in filtered_txts.items():
    print(f"\n{'─'*40}")
    print(f"📄 {name}: {'✅ Existe' if path.exists() else '❌ Não encontrado'}")
    if path.exists():
        with open(path) as f:
            lines = f.readlines()
        print(f"   Total de linhas: {len(lines)}")
        print(f"   Primeiras 5 linhas:")
        for line in lines[:5]:
            print(f"     {line.strip()}")

---
## 4. Análise dos Nomes de Arquivos — Extração do Patient ID
Investigamos o padrão dos filenames para entender como extrair o `patient_id`.

In [ ]:
# Coleta todos os filenames dos arquivos de anotação originais
all_filenames = []

for name, path in {**original_txts, **filtered_txts}.items():
    if path.exists():
        with open(path) as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    all_filenames.append(parts[0])

# Remove duplicatas para análise
unique_filenames = sorted(set(all_filenames))
print(f"Total de filenames únicos encontrados: {len(unique_filenames)}")
print(f"\n📝 Amostra de 20 filenames (primeiros):")
for fn in unique_filenames[:20]:
    print(f"  {fn}")

print(f"\n📝 Amostra de 20 filenames (últimos):")
for fn in unique_filenames[-20:]:
    print(f"  {fn}")

In [ ]:
# Analisa padrões nos nomes dos arquivos
print("=" * 60)
print("ANÁLISE DE PADRÕES NOS FILENAMES")
print("=" * 60)

# Tenta diferentes estratégias de extração de patient_id
strategies = {
    'split_underscore_first': lambda fn: fn.split('_')[0],
    'split_underscore_first2': lambda fn: '_'.join(fn.split('_')[:2]),
    'without_extension_last_part': lambda fn: '_'.join(os.path.splitext(fn)[0].split('_')[:-1]),
    'regex_numeric_prefix': lambda fn: re.match(r'^(\d+)', fn).group(1) if re.match(r'^(\d+)', fn) else fn,
}

sample = unique_filenames[:30]

print(f"\n🔍 Testando estratégias de extração em {len(sample)} filenames:\n")
for strategy_name, extract_fn in strategies.items():
    try:
        ids = [extract_fn(fn) for fn in sample]
        n_unique = len(set(ids))
        print(f"  [{strategy_name}]")
        print(f"    IDs únicos: {n_unique} (de {len(sample)} filenames)")
        print(f"    Exemplos: {list(set(ids))[:10]}")
        print()
    except Exception as e:
        print(f"  [{strategy_name}] ERRO: {e}\n")

---
## 5. Mapeamento Filename → Patient ID (usando metadata.csv se disponível)
Se o `metadata.csv` existir, construímos o mapeamento oficial. Caso contrário, usamos a melhor heurística de extração.

In [ ]:
def build_patient_mapping(base_path):
    """
    Constrói o mapeamento filename → patient_id.
    Tenta metadata.csv primeiro, depois fallback por heurística.
    Retorna: dict {filename: patient_id}
    """
    metadata_path = base_path / 'metadata.csv'
    
    if metadata_path.exists():
        print("✅ Usando metadata.csv para mapeamento")
        meta = pd.read_csv(metadata_path)
        
        # Encontra a coluna de patient id
        patient_col = None
        filename_col = None
        for col in meta.columns:
            col_lower = col.lower().strip()
            if 'patient' in col_lower and 'id' in col_lower:
                patient_col = col
            if 'filename' in col_lower or 'file' in col_lower:
                filename_col = col
        
        print(f"   Coluna de paciente: {patient_col}")
        print(f"   Coluna de filename: {filename_col}")
        print(f"   Todas as colunas: {list(meta.columns)}")
        
        if patient_col and filename_col:
            mapping = dict(zip(meta[filename_col].astype(str), meta[patient_col].astype(str)))
            return mapping, 'metadata_csv'
        elif patient_col:
            # Talvez não tenha filename, mas tenha outra forma de mapear
            print(f"\n   ⚠️ Sem coluna de filename direta. Colunas disponíveis: {list(meta.columns)}")
            print(f"   Mostrando primeiras linhas para análise manual:")
            display(meta.head(10))
            return None, 'metadata_csv_incomplete'
    
    # Fallback: extrai do nome do arquivo
    print("⚠️ metadata.csv não disponível. Usando heurística de extração do filename.")
    print("   ATENÇÃO: Verifique se a heurística está correta para o seu dataset!")
    return None, 'heuristic_needed'

mapping, method = build_patient_mapping(BASE)

if mapping:
    print(f"\n📊 Mapeamento construído: {len(mapping)} entradas")
    # Mostra algumas amostras
    sample_items = list(mapping.items())[:10]
    print(f"\n   Amostra (filename → patient_id):")
    for fn, pid in sample_items:
        print(f"     {fn:>40} → {pid}")

---
## 6. Análise de Overlap de Pacientes entre Splits
Este é o teste mais importante: **existem pacientes que aparecem em mais de um split (treino/val/teste)?** Se sim, temos data leakage confirmado.

In [ ]:
def load_split_filenames(txt_path):
    """Carrega os filenames de um arquivo de anotação."""
    if not txt_path.exists():
        return []
    with open(txt_path) as f:
        return [line.strip().split()[0] for line in f if line.strip()]

def get_patient_id(filename, mapping=None):
    """Obtém o patient_id de um filename."""
    if mapping and filename in mapping:
        return mapping[filename]
    # Fallback: primeiro segmento antes do underscore
    return filename.split('_')[0]

# --- Análise nos SPLITS ORIGINAIS do COVIDx CT-3A ---
print("=" * 60)
print("ANÁLISE DE OVERLAP — SPLITS ORIGINAIS")
print("=" * 60)

original_splits = {}
for name, path in original_txts.items():
    fns = load_split_filenames(path)
    if fns:
        pids = set(get_patient_id(fn, mapping) for fn in fns)
        original_splits[name] = {'filenames': fns, 'patient_ids': pids}
        print(f"  {name:>20}: {len(fns):>6} slices | {len(pids):>5} pacientes")

# Verifica overlap entre todos os pares
if len(original_splits) >= 2:
    split_names = list(original_splits.keys())
    print(f"\n🔍 Verificação de overlap de pacientes:")
    has_leakage = False
    for i, name1 in enumerate(split_names):
        for name2 in split_names[i+1:]:
            overlap = original_splits[name1]['patient_ids'] & original_splits[name2]['patient_ids']
            status = '❌ LEAKAGE!' if overlap else '✅ OK (disjoint)'
            print(f"  {name1} ∩ {name2}: {len(overlap)} pacientes em comum — {status}")
            if overlap:
                has_leakage = True
                print(f"    Exemplos de pacientes em overlap: {list(overlap)[:10]}")
    
    if not has_leakage:
        print("\n  ✅ O split original do COVIDx CT-3A já é patient-disjoint!")
    else:
        print("\n  ❌ O split original TEM data leakage entre pacientes!")

In [ ]:
# --- Análise nos SPLITS FILTRADOS (gerados pelo prepare_filtered_data.py) ---
print("=" * 60)
print("ANÁLISE DE OVERLAP — SPLITS FILTRADOS (ATUAIS)")
print("=" * 60)

filtered_splits = {}
for name, path in filtered_txts.items():
    fns = load_split_filenames(path)
    if fns:
        pids = set(get_patient_id(fn, mapping) for fn in fns)
        filtered_splits[name] = {'filenames': fns, 'patient_ids': pids}
        print(f"  {name:>20}: {len(fns):>6} slices | {len(pids):>5} pacientes")

if len(filtered_splits) >= 2:
    split_names = list(filtered_splits.keys())
    print(f"\n🔍 Verificação de overlap de pacientes:")
    for i, name1 in enumerate(split_names):
        for name2 in split_names[i+1:]:
            overlap = filtered_splits[name1]['patient_ids'] & filtered_splits[name2]['patient_ids']
            status = '❌ LEAKAGE!' if overlap else '✅ OK (disjoint)'
            print(f"  {name1} ∩ {name2}: {len(overlap)} pacientes em comum — {status}")
            if overlap:
                print(f"    Exemplos: {list(overlap)[:10]}")
                
                # Quantifica o impacto
                n_leaked_slices_1 = sum(1 for fn in filtered_splits[name1]['filenames'] if get_patient_id(fn, mapping) in overlap)
                n_leaked_slices_2 = sum(1 for fn in filtered_splits[name2]['filenames'] if get_patient_id(fn, mapping) in overlap)
                print(f"    Slices afetados em {name1}: {n_leaked_slices_1}")
                print(f"    Slices afetados em {name2}: {n_leaked_slices_2}")

---
## 7. Distribuição de Slices por Paciente
Quantos slices cada paciente tem? Isso impacta diretamente a estratégia de agregação.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

# Coleta todos os filenames de todos os splits disponíveis
all_fns = []
source = filtered_splits if filtered_splits else original_splits

for name, data in source.items():
    all_fns.extend(data['filenames'])

# Conta slices por paciente
patient_slice_counts = Counter(get_patient_id(fn, mapping) for fn in all_fns)

counts = list(patient_slice_counts.values())
print(f"📊 Estatísticas de slices por paciente:")
print(f"   Total de pacientes: {len(counts)}")
print(f"   Total de slices: {sum(counts)}")
print(f"   Mínimo: {min(counts)}")
print(f"   Máximo: {max(counts)}")
print(f"   Média: {np.mean(counts):.1f}")
print(f"   Mediana: {np.median(counts):.0f}")
print(f"   Percentil 25: {np.percentile(counts, 25):.0f}")
print(f"   Percentil 75: {np.percentile(counts, 75):.0f}")
print(f"   Percentil 95: {np.percentile(counts, 95):.0f}")

# Histograma
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(counts, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_title('Distribuição de Slices por Paciente', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Nº de Slices')
axes[0].set_ylabel('Nº de Pacientes')
axes[0].axvline(np.mean(counts), color='red', linestyle='--', label=f'Média: {np.mean(counts):.1f}')
axes[0].axvline(np.median(counts), color='green', linestyle='--', label=f'Mediana: {np.median(counts):.0f}')
axes[0].legend()

# Boxplot
axes[1].boxplot(counts, vert=True)
axes[1].set_title('Boxplot de Slices por Paciente', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Nº de Slices')

plt.tight_layout()
plt.show()

# Top pacientes com mais slices
top_patients = patient_slice_counts.most_common(15)
print(f"\n🔝 Top 15 pacientes com mais slices:")
for pid, count in top_patients:
    print(f"   Patient {pid}: {count} slices")

---
## 8. Distribuição de Classes por Paciente
Cada paciente tem um único diagnóstico (todos os slices dele têm a mesma classe)? Isso é crucial para saber se a agregação por votação faz sentido.

In [ ]:
# Para cada split, agrupa por paciente e verifica se todos os slices têm a mesma classe
def load_entries(txt_path):
    """Carrega (filename, label) de um .txt"""
    if not txt_path.exists():
        return []
    entries = []
    with open(txt_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 2:
                entries.append((parts[0], int(parts[1])))
    return entries

print("=" * 60)
print("CONSISTÊNCIA DE CLASSES POR PACIENTE")
print("=" * 60)

# Usa os arquivos filtrados (que já removeram a classe Normal)
all_entries = []
for path in filtered_txts.values():
    all_entries.extend(load_entries(path))

# Se não tem filtrados, usa os originais
if not all_entries:
    for path in original_txts.values():
        all_entries.extend(load_entries(path))

# Agrupa por paciente
patient_labels = {}
for fn, label in all_entries:
    pid = get_patient_id(fn, mapping)
    patient_labels.setdefault(pid, set()).add(label)

# Verifica inconsistências
inconsistent = {pid: labels for pid, labels in patient_labels.items() if len(labels) > 1}

print(f"\n  Pacientes com classe única: {len(patient_labels) - len(inconsistent)}")
print(f"  Pacientes com classes mistas: {len(inconsistent)}")

if inconsistent:
    print(f"\n  ⚠️ PACIENTES COM CLASSES MISTAS (slices com diagnósticos diferentes):")
    for pid, labels in list(inconsistent.items())[:20]:
        # Conta quantos slices de cada classe
        label_counts = Counter(label for fn, label in all_entries if get_patient_id(fn, mapping) == pid)
        print(f"    Patient {pid}: classes {labels} — distribuição: {dict(label_counts)}")
else:
    print(f"\n  ✅ Todos os pacientes têm classe consistente (todos os slices com mesmo diagnóstico)")
    print(f"     Isso confirma que a agregação por votação majoritária ou média fará sentido.")

---
## 9. Resumo e Recomendações
Consolidação de todos os achados para guiar a reestruturação.

In [ ]:
print("\n" + "=" * 60)
print("📋 RESUMO DO DIAGNÓSTICO")
print("=" * 60)

print(f"\n1. metadata.csv: {'✅ Disponível' if has_metadata else '❌ Não encontrado'}")
print(f"   → Método de mapeamento: {method}")

print(f"\n2. Pacientes totais: {len(patient_labels)}")
print(f"   Slices totais: {len(all_entries)}")
print(f"   Média slices/paciente: {len(all_entries)/len(patient_labels):.1f}" if patient_labels else "   N/A")

print(f"\n3. Consistência de classes: {'✅ OK' if not inconsistent else f'⚠️ {len(inconsistent)} pacientes com classes mistas'}")

# Overlap nos filtrados
if filtered_splits and len(filtered_splits) >= 2:
    any_leak = False
    for i, name1 in enumerate(list(filtered_splits.keys())):
        for name2 in list(filtered_splits.keys())[i+1:]:
            overlap = filtered_splits[name1]['patient_ids'] & filtered_splits[name2]['patient_ids']
            if overlap:
                any_leak = True
    print(f"\n4. Data leakage nos splits filtrados: {'❌ SIM — CONFIRMADO' if any_leak else '✅ NÃO (patient-disjoint)'}")

# Overlap nos originais
if original_splits and len(original_splits) >= 2:
    any_leak_orig = False
    for i, name1 in enumerate(list(original_splits.keys())):
        for name2 in list(original_splits.keys())[i+1:]:
            overlap = original_splits[name1]['patient_ids'] & original_splits[name2]['patient_ids']
            if overlap:
                any_leak_orig = True
    print(f"   Data leakage nos splits originais: {'❌ SIM' if any_leak_orig else '✅ NÃO (patient-disjoint)'}")

print(f"\n{'='*60}")
print("📝 PRÓXIMOS PASSOS")
print("=" * 60)
print("""
Com base nos resultados acima, precisamos decidir:

1. Se o metadata.csv existe e tem o mapeamento filename → patient_id,
   usaremos ele. Caso contrário, confirme a heurística de extração.

2. Se os splits originais JÁ são patient-disjoint (train vs test),
   devemos respeitá-los e apenas refazer o sub-split train → train/val
   agrupando por paciente.

3. Se os splits filtrados TÊM overlap, isso confirma o data leakage
   e a necessidade urgente da reestruturação.

4. A distribuição de slices por paciente vai guiar a estratégia de
   agregação (votação majoritária vs média de probabilidades).
""")